In [ ]:
import glob
import os
import re

import pandas as pd


def concatenate_csvs(input_dir="../data/input", output_dir="../data/output"):
    """
    Concatenate autofolio CSV files, filtering for video mimeTypes and adding source agency names.

    Args:
        input_dir: Directory containing input CSV files
        output_dir: Directory for output CSV file
    """

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Find all CSV files matching the pattern
    csv_files = glob.glob(os.path.join(input_dir, "autofolio_*_output--*.csv"))

    if not csv_files:
        print("No matching CSV files found!")
        return

    all_dataframes = []

    for csv_file in csv_files:
        try:
            # Extract agency name from filename
            filename = os.path.basename(csv_file)

            # Pattern: autofolio_*_output--{AGENCY_NAME}--{DATE}
            match = re.search(r"output--(.+?)--\d{4}-\d{2}-\d{2}", filename)

            if not match:
                print(f"Warning: Could not extract agency name from {filename}")
                continue

            agency_name = match.group(1)

            # Convert to lowercase and replace spaces with underscores
            source_agency_name = agency_name.lower().replace(" ", "_")

            # Read CSV
            df = pd.read_csv(csv_file)

            # Check if mimeType column exists
            if "mimeType" not in df.columns:
                print(f"Warning: mimeType column not found in {filename}")
                continue

            # Add source agency name column
            df["source_agency_name"] = source_agency_name

            all_dataframes.append(df)
            print(f"Processed {filename}: {len(df)} video records from {agency_name}")

        except Exception as e:
            print(f"Error processing {csv_file}: {str(e)}")
            continue

    if not all_dataframes:
        print("No data to concatenate!")
        return

    # Concatenate all dataframes
    final_df = pd.concat(all_dataframes, ignore_index=True)

    # Save to output directory
    output_file = os.path.join(output_dir, "autofolio_df_concat.csv")
    final_df.to_csv(output_file, index=False)

    print("\nConcatenation complete!")
    print(f"Total records: {len(final_df)}")
    print(f"Output saved to: {output_file}")
    print(f"Agencies included: {final_df['source_agency_name'].nunique()}")


concatenate_csvs()